# All_Program_Enrollments

This all_program_enrollments script is foundational to the [[ORGANIZATION_NAME]] Data Hub. As seen in the entity relationship diagram: [[ORG_ERD_URL]], most of the analytic tables in the [[ORGANIZATION_NAME]] Data Hub link to the output of this script, the all_program_enrollments enrollments table. 

The code brings in enrollment level data for all programs in our HMIS with an active enrollment in the year 2017 or onward. The script transforms the data to clean HMIS data and to add custom categorizations and rollups specific to [[ORGANIZATION_NAME]].


In [ ]:
import looker_sdk
from looker_sdk import api_settings
from io import StringIO
import pandas as pd
from datetime import datetime, timedelta
from datetime import datetime
from datetime import date
import numpy as np 
from azure.storage.blob import BlobServiceClient
from io import BytesIO

# Import Looker SDK connection from shared script (for development)
from [[LOCAL_SCRIPT_NAME]] import sdk

# # For production in Azure, comment out the import above and use this instead:
# class [[ORG_PREFIX]]_Looker_API_Settings(api_settings.ApiSettings):
#     def __init__(self, *args, **kw_args):
#         self.my_var = kw_args.pop("my_var")
#         super().__init__(*args, **kw_args)

#     def read_config(self) -> api_settings.SettingsConfig:
#         config = super().read_config()
#         # See api_settings.SettingsConfig for required fields.
#         if self.my_var == "set":
#             config["base_url"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_BASE_URL_SECRET]]')
#             config["client_id"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_CLIENT_ID_SECRET]]')
#             config["client_secret"] = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]','[[LOOKER_CLIENT_SECRET_SECRET]]')          
#         return config

# sdk = looker_sdk.init40(config_settings=[[ORG_PREFIX]]_Looker_API_Settings(my_var="set"))
# vars(vars(vars(sdk)["transport"])["settings"])["timeout"] = 1000

In [ ]:
# Define your query with concurrent requests and exponential backoff

from concurrent.futures import ThreadPoolExecutor, as_completed
import time

seven_years_ago = date.today().replace(year=date.today().year - 7, month=1, day=1).isoformat()

# Base fields (not redefined in each loop)
fields = [
    "enrollments.id",
    "enrollments.start_date",
    "household_move_in_date.move_in_date",
    "enrollments.end_date",
    "programs.project_type_code",
    "programs.agency_name",
    "programs.agency_id",
    "programs.name",
    "programs.id",
    "clients.unique_identifier",
    "clients.personal_id",
    "clients.id",
    "entry_screen.age_tier",
    "entry_screen.age",
    "clients.consent_refused",
    "enrollments.ref_household",
    "entry_screen.head_of_household",
    "last_screen.exit_destination_text",
    "last_screen.exit_destination_category",
    "chronically_homeless_households.is_chronic_homeless_household",
    "chronically_homeless_at_entry.is_chronic_homeless",
    "household_makeup.count_adults",
    "household_makeup.oldest",
    "household_makeup.count_children",
    "entry_screen.relationship_to_hoh",
    "entry_screen.prior_residence_category",
    "entry_screen.prior_residence_text",
    "entry_screen.health_dv_fleeing",
    "household_entry_screen.area_median_income",
    "household_exit_screen.area_median_income"
]

RESULT_FORMAT = "csv"

# All the project type codes you want to pull
all_codes = list(range(15))

# Retry settings with exponential backoff
max_retries = 3
base_delay_seconds = 2  # Will be 2, 4, 8 seconds for retries 1, 2, 3

def fetch_project_type(code):
    """Fetch data for a single project type code with exponential backoff retries."""
    body = {
        "model": "[[LOOKER_MODEL_NAME]]",
        "view": "base",
        "fields": fields,
        "filters": {
            "enrollments.date_filter": "NULL",
            "enrollments.end_date_or_today_date": f"after {seven_years_ago}",
            "programs.raw_project_type_code": str(code),
            "program_hmis_participation_statuses.hmis_participation_status":
                "'HMIS Participating' , 'Comparable Database Participating'"
        },
        "limit": -1
    }
    
    for attempt in range(1, max_retries + 1):
        try:
            result = sdk.run_inline_query(
                result_format=RESULT_FORMAT,
                body=body
            )
            df = pd.read_csv(StringIO(result))
            print(f"✔ Success for code {code} on attempt {attempt}")
            return code, df, None
        except Exception as e:
            if attempt < max_retries:
                delay = base_delay_seconds * (2 ** (attempt - 1))  # Exponential backoff
                print(f"⚠ Error for code {code}, attempt {attempt}/{max_retries}: {e}. Retrying in {delay}s...")
                time.sleep(delay)
            else:
                print(f"✖ Failed for code {code} after {max_retries} attempts: {e}")
                return code, None, str(e)

# Run concurrent requests (3 workers to avoid overwhelming the Looker API)
max_workers = 3
results = {}
failed_codes = []

print(f"Starting concurrent fetch with {max_workers} workers for {len(all_codes)} project type codes...")

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(fetch_project_type, code): code for code in all_codes}
    
    for future in as_completed(futures):
        code, df, error = future.result()
        if df is not None:
            results[code] = df
        else:
            failed_codes.append((code, error))

# Report results
print(f"\nCompleted: {len(results)} codes")
if failed_codes:
    print(f"✖ WARNING: The following project type codes failed: {[c for c, e in failed_codes]}")

if not results:
    raise RuntimeError("No dataframes were returned; cannot concatenate.")

# Concatenate all successful results
df0 = pd.concat(results.values(), ignore_index=True)

print(f"Total rows retrieved: {len(df0)}")
print("Completed codes:", sorted(results.keys()))

In [ ]:
#add timestamp
# Get today's date
today_date = datetime.today().date()

# Assign the date to the DataFrame
df0['DataAsOfDate'] = today_date

In [ ]:
df = df0

Create Dictionary for Script

In [ ]:
columnHeaders={ "Enrollments Enrollment ID" : 'EnrollmentID',
    "Enrollments Project Start Date": "ProjectStartDate",
    "Enrollments Household Move-In Date": "HouseholdMoveInDate",
    "Enrollments Project Exit Date": "ProjectExitDate",
    "Programs Project Type Code": "ProjectTypeCode",
    "Programs Agency Name": "AgencyName",
    "Programs Agency ID": "AgencyID",
    "Programs Name": "ProgramName",
    "Programs Program ID": "ProgramID",
    "Clients Unique Identifier": "ClientUniqueIdentifier",
    "Clients Personal ID": "PersonalID",
    "Clients Client ID": "ClientID",
    "Entry Screen Age Tier": "AgeTierAtEnrollment",
    "Entry Screen Age at Project Start" : "AgeAtEnrollment",
    "Clients Consent Refused (Yes / No)": "ConsentRefused",
    "Enrollments Household ID": "HouseholdID",
    "Entry Screen Head of Household (Yes / No)": "HeadOfHousehold",
    "Update/Exit Screen Destination": 'ExitDestination', 
    "Update/Exit Screen Destination Category": "ExitDestinationCategory",
    "Entry Screen Chronically Homeless Project Start - Household": "HouseholdChronicallyHomelessAtEnrollmentStart",
    "Entry Screen Chronically Homeless at Project Start - Individual": "IndividualChronicallyHomelessAtEnrollmentStart",
    "Enrollments Count Adults": "CountAdults",
    "Enrollments Count Children": "CountChildren",
    "Enrollments Age of Oldest HH Member": "OldestHouseholdMember",
    'Entry Screen Relationship to Head of Household':'RelationshipToHeadOfHousehold',
    "Entry Screen Prior Living Situation Category": "PriorResidenceCategory",
    "Entry Screen Residence Prior to Project Entry" :"PriorResidence",
    "DataAsOfDate" : "DataAsOfDate",
    "Entry Screen Currently Fleeing Domestic Violence": "CurrentlyFleeing",
    "Entry Screen Area Median Income" : "EntryAMI",
    "Update/Exit Screen Area Median Income" : "ExitAMI"
}

df.rename(columns=columnHeaders,inplace=True)

Cleaning up Head of Household IDs

In [ ]:
# Count the number of distinct 'ClientUniqueIdentifier' for 'Self (head of household)' per HouseholdID
self_counts = df[df['RelationshipToHeadOfHousehold'] == 'Self (head of household)'].groupby('HouseholdID')['ClientUniqueIdentifier'].nunique()

# Count the number of HouseholdIDs with 0, 1, and 2 or more distinct 'Self (head of household)' records
household_counts = df.groupby('HouseholdID').size()
self_zero = household_counts[~household_counts.index.isin(self_counts.index)].count()
self_one = self_counts[self_counts == 1].count()
self_two_or_more = self_counts[self_counts >= 2].count()

# Print the results
print(f"Number of HouseholdIDs with 0 'Self (head of household)' records: {self_zero}")
print(f"Number of HouseholdIDs with 1 'Self (head of household)' record: {self_one}")
print(f"Number of HouseholdIDs with 2 or more 'Self (head of household)' records: {self_two_or_more}")

self_two_or_more_list = self_counts[self_counts >= 2].index.tolist()  # Convert index to list of HouseholdIDs
self_zero_list = household_counts[~household_counts.index.isin(self_counts.index)].index.tolist()  # Convert index to list of HouseholdIDs


In [ ]:
# Initialize the new column for the clean household head designation
df['HouseholdHeadClean'] = df['RelationshipToHeadOfHousehold']

# Group by HouseholdID
household_groups = df.groupby('HouseholdID')

def clean_household_heads(group):
    # Check if there is exactly one 'Self (head of household)'
    head_count = group[group['RelationshipToHeadOfHousehold'] == 'Self (head of household)'].shape[0]
    
    # 1. Check for exactly one identified household head
    if head_count == 1:
        # Find the index of the household head
        head_idx = group[group['RelationshipToHeadOfHousehold'] == 'Self (head of household)'].index[0]
        
        # Check if the household head is under 18 and there is someone older
        if group.loc[head_idx, 'AgeAtEnrollment'] < 18:
            oldest_member_idx = group['AgeAtEnrollment'].idxmax()
            group.loc[head_idx, 'HouseholdHeadClean'] = 'Other'  # Change current head to 'Other'
            group.loc[oldest_member_idx, 'HouseholdHeadClean'] = 'Self (head of household)'  # Assign oldest member as head
        
    # 2. Handle cases with no identified household heads
    elif head_count == 0:
        if group['AgeAtEnrollment'].isnull().all():
            first_member_idx = group.index[0]  # Pick the first record if all ages are null
            group.loc[first_member_idx, 'HouseholdHeadClean'] = 'Self (head of household)'
        else:
            oldest_member_idx = group['AgeAtEnrollment'].idxmax()
            group.loc[oldest_member_idx, 'HouseholdHeadClean'] = 'Self (head of household)'
    
    # 3. Handle cases with multiple identified household heads
    else:
        if group['AgeAtEnrollment'].isnull().all():
            first_member_idx = group.index[0]  # Pick the first record if all ages are null
            group['HouseholdHeadClean'] = group['RelationshipToHeadOfHousehold'].apply(lambda x: 'Other' if x == 'Self (head of household)' else x)
            group.loc[first_member_idx, 'HouseholdHeadClean'] = 'Self (head of household)'
        else:
            oldest_head_idx = group[group['RelationshipToHeadOfHousehold'] == 'Self (head of household)']['AgeAtEnrollment'].idxmax()
            group['HouseholdHeadClean'] = group['RelationshipToHeadOfHousehold'].apply(lambda x: 'Other' if x == 'Self (head of household)' else x)
            group.loc[oldest_head_idx, 'HouseholdHeadClean'] = 'Self (head of household)'
    
    return group

# Apply the function to each household group
df_cleaned = household_groups.apply(clean_household_heads)


In [ ]:
df_cleaned = df_cleaned.reset_index(drop=True)

# Count the number of distinct 'ClientUniqueIdentifier' for 'Self (head of household)' per HouseholdID
self_counts = df_cleaned[df_cleaned['HouseholdHeadClean'] == 'Self (head of household)'].groupby('HouseholdID')['ClientUniqueIdentifier'].nunique()

# Count the number of HouseholdIDs with 0, 1, and 2 or more distinct 'Self (head of household)' records
household_counts = df_cleaned.groupby('HouseholdID').size()
self_zero = household_counts[~household_counts.index.isin(self_counts.index)].count()
self_one = self_counts[self_counts == 1].count()
self_two_or_more = self_counts[self_counts >= 2].count()

# Print the results
print(f"Number of HouseholdIDs with 0 'Self (head of household)' records: {self_zero}")
print(f"Number of HouseholdIDs with 1 'Self (head of household)' record: {self_one}")
print(f"Number of HouseholdIDs with 2 or more 'Self (head of household)' records: {self_two_or_more}")

# Create qa datasets for HouseholdIDs that were missing or duplicates and see how they are coded in the clean version
selected_columns = ['HouseholdID', 'HeadOfHousehold', 'RelationshipToHeadOfHousehold', 'HouseholdHeadClean', 'AgeAtEnrollment', 'OldestHouseholdMember', 'CountAdults', 'CountChildren', 'ClientUniqueIdentifier', 'AgencyName']
qa_zero = df_cleaned[df_cleaned['HouseholdID'].isin(self_zero_list)][selected_columns]
qa_multiple = df_cleaned[df_cleaned['HouseholdID'].isin(self_two_or_more_list)][selected_columns]

# Count the number of HouseholdIDs with 0 'Self (head of household)' records
household_counts = df_cleaned.groupby('HouseholdID').size()
self_counts = df_cleaned[df_cleaned['HouseholdHeadClean'] == 'Self (head of household)'].groupby('HouseholdID')['ClientUniqueIdentifier'].nunique()
self_zero = household_counts[~household_counts.index.isin(self_counts.index)].index.tolist()

# Filter df_cleaned by self_zero HouseholdIDs
self_zero_filtered = df_cleaned[df_cleaned['HouseholdID'].isin(self_zero)]

Apply the household head cleaning logic to the dataframe

In [ ]:
# Create the new column 'HeadOfHousehold_clean'
df_cleaned['HeadOfHousehold_clean'] = df_cleaned['HouseholdHeadClean'].apply(lambda x: 'Yes' if x == 'Self (head of household)' else 'No')

# Drop the specified columns
df_cleaned = df_cleaned.drop(columns=['HouseholdHeadClean', 'HeadOfHousehold'])

# Rename 'HeadOfHousehold_clean' to 'HeadOfHousehold'
df_cleaned = df_cleaned.rename(columns={'HeadOfHousehold_clean': 'HeadOfHousehold'})

# Overwrite the existing df with df_cleaned
df = df_cleaned

Household Type Rollup

In [ ]:
#
 
A
d
d
 
Y
Y
A
,
 
H
o
u
s
e
h
o
l
d
T
y
p
e
,
 
a
n
d
 
H
o
u
s
e
h
o
l
d
C
a
t
e
g
o
r
y
 
c
o
l
u
m
n
s

#
 
V
e
c
t
o
r
i
z
e
d
 
a
p
p
r
o
a
c
h
 
f
o
r
 
p
e
r
f
o
r
m
a
n
c
e


#
 
C
r
e
a
t
e
 
h
e
l
p
e
r
 
c
o
l
u
m
n
s
 
f
o
r
 
a
g
e
 
c
a
t
e
g
o
r
i
e
s

d
f
[
'
_
i
s
_
y
o
u
t
h
_
a
g
e
'
]
 
=
 
(
d
f
[
'
A
g
e
A
t
E
n
r
o
l
l
m
e
n
t
'
]
 
>
=
 
1
2
)
 
&
 
(
d
f
[
'
A
g
e
A
t
E
n
r
o
l
l
m
e
n
t
'
]
 
<
=
 
2
4
)

d
f
[
'
_
i
s
_
2
5
_
p
l
u
s
'
]
 
=
 
d
f
[
'
A
g
e
A
t
E
n
r
o
l
l
m
e
n
t
'
]
 
>
=
 
2
5

d
f
[
'
_
i
s
_
c
h
i
l
d
'
]
 
=
 
d
f
[
'
A
g
e
A
t
E
n
r
o
l
l
m
e
n
t
'
]
 
<
 
1
8

d
f
[
'
_
i
s
_
a
d
u
l
t
'
]
 
=
 
d
f
[
'
A
g
e
A
t
E
n
r
o
l
l
m
e
n
t
'
]
 
>
=
 
1
8

d
f
[
'
_
a
g
e
_
k
n
o
w
n
'
]
 
=
 
d
f
[
'
A
g
e
A
t
E
n
r
o
l
l
m
e
n
t
'
]
.
n
o
t
n
a
(
)


#
 
A
g
g
r
e
g
a
t
e
 
b
y
 
h
o
u
s
e
h
o
l
d

h
o
u
s
e
h
o
l
d
_
a
g
g
 
=
 
d
f
.
g
r
o
u
p
b
y
(
'
H
o
u
s
e
h
o
l
d
I
D
'
)
.
a
g
g
(

 
 
 
 
h
a
s
_
y
o
u
t
h
=
(
'
_
i
s
_
y
o
u
t
h
_
a
g
e
'
,
 
'
a
n
y
'
)
,

 
 
 
 
h
a
s
_
2
5
_
p
l
u
s
=
(
'
_
i
s
_
2
5
_
p
l
u
s
'
,
 
'
a
n
y
'
)
,

 
 
 
 
h
a
s
_
c
h
i
l
d
=
(
'
_
i
s
_
c
h
i
l
d
'
,
 
'
a
n
y
'
)
,

 
 
 
 
h
a
s
_
a
d
u
l
t
=
(
'
_
i
s
_
a
d
u
l
t
'
,
 
'
a
n
y
'
)
,

 
 
 
 
a
l
l
_
a
g
e
s
_
u
n
k
n
o
w
n
=
(
'
_
a
g
e
_
k
n
o
w
n
'
,
 
l
a
m
b
d
a
 
x
:
 
~
x
.
a
n
y
(
)
)
,

 
 
 
 
h
a
s
_
u
n
k
n
o
w
n
_
a
g
e
=
(
'
_
a
g
e
_
k
n
o
w
n
'
,
 
l
a
m
b
d
a
 
x
:
 
~
x
.
a
l
l
(
)
)

)
.
r
e
s
e
t
_
i
n
d
e
x
(
)


#
 
Y
Y
A
:
 
a
t
 
l
e
a
s
t
 
o
n
e
 
1
2
-
2
4
,
 
n
o
 
o
n
e
 
2
5
+
,
 
a
n
d
 
a
t
 
l
e
a
s
t
 
o
n
e
 
k
n
o
w
n
 
a
g
e

h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
Y
Y
A
'
]
 
=
 
n
p
.
w
h
e
r
e
(

 
 
 
 
~
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
a
l
l
_
a
g
e
s
_
u
n
k
n
o
w
n
'
]
 
&
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
y
o
u
t
h
'
]
 
&
 
~
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
2
5
_
p
l
u
s
'
]
,

 
 
 
 
'
Y
e
s
'
,
 
'
N
o
'

)


#
 
H
o
u
s
e
h
o
l
d
T
y
p
e
 
d
e
t
e
r
m
i
n
a
t
i
o
n
 
(
a
p
p
l
i
e
d
 
i
n
 
p
r
i
o
r
i
t
y
 
o
r
d
e
r
)

c
o
n
d
i
t
i
o
n
s
 
=
 
[

 
 
 
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
c
h
i
l
d
'
]
 
&
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
a
d
u
l
t
'
]
,

 
 
 
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
a
d
u
l
t
'
]
 
&
 
~
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
c
h
i
l
d
'
]
 
&
 
~
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
u
n
k
n
o
w
n
_
a
g
e
'
]
,

 
 
 
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
c
h
i
l
d
'
]
 
&
 
~
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
a
d
u
l
t
'
]
 
&
 
~
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
h
a
s
_
u
n
k
n
o
w
n
_
a
g
e
'
]

]

c
h
o
i
c
e
s
 
=
 
[

 
 
 
 
"
H
o
u
s
e
h
o
l
d
 
w
i
t
h
 
C
h
i
l
d
r
e
n
 
a
n
d
 
A
d
u
l
t
s
"
,

 
 
 
 
"
H
o
u
s
e
h
o
l
d
 
w
i
t
h
o
u
t
 
C
h
i
l
d
r
e
n
"
,

 
 
 
 
"
H
o
u
s
e
h
o
l
d
 
w
i
t
h
 
O
n
l
y
 
C
h
i
l
d
r
e
n
"

]

h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
H
o
u
s
e
h
o
l
d
T
y
p
e
'
]
 
=
 
n
p
.
s
e
l
e
c
t
(
c
o
n
d
i
t
i
o
n
s
,
 
c
h
o
i
c
e
s
,
 
d
e
f
a
u
l
t
=
"
U
n
k
n
o
w
n
 
H
o
u
s
e
h
o
l
d
 
T
y
p
e
"
)


#
 
H
o
u
s
e
h
o
l
d
C
a
t
e
g
o
r
y
 
d
e
r
i
v
a
t
i
o
n
 
(
a
p
p
l
i
e
d
 
i
n
 
p
r
i
o
r
i
t
y
 
o
r
d
e
r
)

c
a
t
_
c
o
n
d
i
t
i
o
n
s
 
=
 
[

 
 
 
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
H
o
u
s
e
h
o
l
d
T
y
p
e
'
]
 
=
=
 
"
H
o
u
s
e
h
o
l
d
 
w
i
t
h
 
C
h
i
l
d
r
e
n
 
a
n
d
 
A
d
u
l
t
s
"
,

 
 
 
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
Y
Y
A
'
]
 
=
=
 
"
Y
e
s
"
,

 
 
 
 
h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
H
o
u
s
e
h
o
l
d
T
y
p
e
'
]
 
=
=
 
"
H
o
u
s
e
h
o
l
d
 
w
i
t
h
o
u
t
 
C
h
i
l
d
r
e
n
"

]

c
a
t
_
c
h
o
i
c
e
s
 
=
 
[

 
 
 
 
"
F
a
m
i
l
y
 
w
i
t
h
 
C
h
i
l
d
r
e
n
"
,

 
 
 
 
"
Y
o
u
t
h
 
a
n
d
 
Y
o
u
n
g
 
A
d
u
l
t
s
"
,

 
 
 
 
"
I
n
d
i
v
i
d
u
a
l
 
A
d
u
l
t
s
"

]

h
o
u
s
e
h
o
l
d
_
a
g
g
[
'
H
o
u
s
e
h
o
l
d
C
a
t
e
g
o
r
y
'
]
 
=
 
n
p
.
s
e
l
e
c
t
(
c
a
t
_
c
o
n
d
i
t
i
o
n
s
,
 
c
a
t
_
c
h
o
i
c
e
s
,
 
d
e
f
a
u
l
t
=
"
O
t
h
e
r
"
)


#
 
C
r
e
a
t
e
 
m
a
p
p
i
n
g
 
d
i
c
t
i
o
n
a
r
i
e
s

y
y
a
_
m
a
p
 
=
 
h
o
u
s
e
h
o
l
d
_
a
g
g
.
s
e
t
_
i
n
d
e
x
(
'
H
o
u
s
e
h
o
l
d
I
D
'
)
[
'
Y
Y
A
'
]
.
t
o
_
d
i
c
t
(
)

h
o
u
s
e
h
o
l
d
_
t
y
p
e
_
m
a
p
 
=
 
h
o
u
s
e
h
o
l
d
_
a
g
g
.
s
e
t
_
i
n
d
e
x
(
'
H
o
u
s
e
h
o
l
d
I
D
'
)
[
'
H
o
u
s
e
h
o
l
d
T
y
p
e
'
]
.
t
o
_
d
i
c
t
(
)

h
o
u
s
e
h
o
l
d
_
c
a
t
e
g
o
r
y
_
m
a
p
 
=
 
h
o
u
s
e
h
o
l
d
_
a
g
g
.
s
e
t
_
i
n
d
e
x
(
'
H
o
u
s
e
h
o
l
d
I
D
'
)
[
'
H
o
u
s
e
h
o
l
d
C
a
t
e
g
o
r
y
'
]
.
t
o
_
d
i
c
t
(
)


#
 
M
a
p
 
v
a
l
u
e
s
 
b
a
c
k
 
t
o
 
t
h
e
 
d
a
t
a
f
r
a
m
e

d
f
[
'
Y
Y
A
'
]
 
=
 
d
f
[
'
H
o
u
s
e
h
o
l
d
I
D
'
]
.
m
a
p
(
y
y
a
_
m
a
p
)

d
f
[
'
H
o
u
s
e
h
o
l
d
T
y
p
e
'
]
 
=
 
d
f
[
'
H
o
u
s
e
h
o
l
d
I
D
'
]
.
m
a
p
(
h
o
u
s
e
h
o
l
d
_
t
y
p
e
_
m
a
p
)

d
f
[
'
H
o
u
s
e
h
o
l
d
C
a
t
e
g
o
r
y
'
]
 
=
 
d
f
[
'
H
o
u
s
e
h
o
l
d
I
D
'
]
.
m
a
p
(
h
o
u
s
e
h
o
l
d
_
c
a
t
e
g
o
r
y
_
m
a
p
)


#
 
C
l
e
a
n
 
u
p
 
h
e
l
p
e
r
 
c
o
l
u
m
n
s

d
f
 
=
 
d
f
.
d
r
o
p
(
c
o
l
u
m
n
s
=
[
'
_
i
s
_
y
o
u
t
h
_
a
g
e
'
,
 
'
_
i
s
_
2
5
_
p
l
u
s
'
,
 
'
_
i
s
_
c
h
i
l
d
'
,
 
'
_
i
s
_
a
d
u
l
t
'
,
 
'
_
a
g
e
_
k
n
o
w
n
'
]
)

Income Fields Rollup

Disability fields rollups

Select and name columns

Should reflect analytic table documentation:
[[ANALYTIC_TABLE_DOCS_URL]]


In [ ]:
columns_to_select = ["EnrollmentID", 
                    "ProjectStartDate",
                    "HouseholdMoveInDate",
                    "ProjectExitDate",
                    "ProjectTypeCode",
                    "AgencyName",
                    "AgencyID",
                    'ProgramName',
                    "ProgramID",
                    "ClientUniqueIdentifier",
                    "PersonalID",
                    "ClientID",
                    "AgeTierAtEnrollment",
                    "ConsentRefused",
                    "HouseholdID",
                    "HouseholdType",
                    "HouseholdCategory",
                    "YYA",
                    "HeadOfHousehold",
                    "ExitDestination",
                    "ExitDestinationCategory",
                    "HouseholdChronicallyHomelessAtEnrollmentStart",
                    "IndividualChronicallyHomelessAtEnrollmentStart",
                    'CountAdults',
                    "CountChildren",
                    "PriorResidenceCategory",
                    "PriorResidence",
                    "DataAsOfDate",
                    "CurrentlyFleeing",
                    "EntryAMI",
                    "ExitAMI"
]

In [ ]:
column_names_mapping = {"EnrollmentID" : "EnrollmentID", 
                    "ProjectStartDate" : "ProjectStartDate",
                    "HouseholdMoveInDate" : "HouseholdMoveInDate",
                    "ProjectExitDate" :"ProjectExitDate",
                    "ProjectTypeCode" :"ProjectTypeCode" ,
                    "AgencyName" :"AgencyName" ,
                    "AgencyID" : "AgencyID",
                    'ProgramName' : 'ProgramName',
                    "ProgramID" : "ProgramID",
                    "ClientUniqueIdentifier" : "ClientUniqueIdentifier",
                    "PersonalID" : "PersonalID",
                    "ClientID" : "ClientID",
                    "AgeTierAtEnrollment" : "AgeTierAtEnrollment",
                    "ConsentRefused" : "ConsentRefused",
                    "HouseholdID" : "HouseholdID",
                    "HouseholdType" : "HouseholdType",
                    "HouseholdCategory" :"HouseholdCategory" ,
                    "YYA" : "YYA",
                    "HeadOfHousehold" : "HeadOfHousehold",
                    "ExitDestination" :"ExitDestination" ,
                    "ExitDestinationCategory" : "ExitDestinationCategory",
                    "HouseholdChronicallyHomelessAtEnrollmentStart" : "HouseholdChronicallyHomelessAtEnrollmentStart",
                    "IndividualChronicallyHomelessAtEnrollmentStart" : "IndividualChronicallyHomelessAtEnrollmentStart",
                    'CountAdults' : 'CountAdults',
                    "CountChildren" : "CountChildren",
                    "PriorResidenceCategory" : "PriorResidenceCategory" ,
                    "PriorResidence" : "PriorResidence",
                    "DataAsOfDate" : "DataAsOfDate",
                    "CurrentlyFleeing":"CurrentlyFleeing",
                    "EntryAMI" : "EntryAMI",
                    "ExitAMI" : "ExitAMI"}

final = df[columns_to_select].rename(columns=column_names_mapping)

In [ ]:
# Initiate Parameter Fields.
container_name = "[[AZURE_CONTAINER_NAME]]"
file_name = "[[OUTPUT_PARQUET_FILENAME]]"

# Create a BlobServiceClient object using the connection string
connection_string = TokenLibrary.getSecretWithLS('[[AZURE_KEY_VAULT_NAME]]', '[[ADLS_CONNECTION_SECRET]]')
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Create a BlobClient for the final blob
final_blob_client = blob_service_client.get_blob_client(container=container_name, blob=file_name)

# Save DataFrame to a temporary Parquet file on Azure Blob Storage
with BytesIO() as temp_buffer:
    final.to_parquet(temp_buffer, engine='pyarrow', index=False)
    final_blob_client.upload_blob(temp_buffer.getvalue(), overwrite=True)